In [19]:
# imports used for webscraping
from bs4 import BeautifulSoup
from multiprocess import Pool
import sys, requests, pandas, numpy, string 

# any config/setup for imported libraries
pandas.options.display.max_columns = None
pandas.options.display.max_rows = None

# define function for collecting desired links
def get_links():
    links = [] 
    for char in string.ascii_lowercase: # loop through alphabet to get letters for url
        page = requests.get(f'https://www.basketball-reference.com/players/{char}')
        soup = BeautifulSoup(page.content, 'html.parser')
        player_links = soup.find_all('tr')

        # loop through and record links on current letter's page 
        for i in range(len(player_links)):
            if i == 0:
                continue
            else: 
                player_link = player_links[i].find('a', href=True)['href'] # searches html a tags to find url
                player_url = f'https://www.basketball-reference.com/{player_link}'
                links.append(player_url)
                
    return links

# define function for collecting player information and create dataframe
def scrape_player(url):
    player_page = requests.get(url)
    player_soup = BeautifulSoup(player_page.content, 'html.parser')
    player_name = player_soup.find(id='meta').find("h1").find("span").text
    player_totals = player_soup.find(id="totals")
    player_playoffs_totals = player_soup.find(id="playoffs_totals")

    if player_totals:
        player_df = pandas.read_html(str(player_totals))[0]
        player_df = player_df[player_df['Age'].notna()]
        if "Unnamed: 30" in player_df.columns:
            player_df = player_df.drop('Unnamed: 30', axis=1)
        player_df.insert(0,'Player Name',player_name)
    if player_playoffs_totals:
        player_df2 = pandas.read_html(str(player_playoffs_totals))[0]
        player_df2 = player_df2[player_df2['Age'].notna()]
        if "Unnamed: 30" in player_df2.columns:
            player_df2 = player_df2.drop('Unnamed: 30', axis=1)
        player_df2.insert(0,'Player Name',player_name)
    
    # Extract the players career accolades
    awards = player_soup.find_all(id='bling')
    player_awards = []
    if awards:
        award = awards[0].find_all('a')
        for i in range(len(award)):
            player_awards.append(award[i].text)
    else:
        player_awards = "None"
    
    if player_playoffs_totals:
        all_stats = pandas.concat([player_df,player_df2], axis=0, ignore_index=True)
        for index, row in all_stats.iterrows():
            row["awards"] = player_awards

        return all_stats
    else:
        for index, row in player_df.iterrows():
            row["awards"] = player_awards
            
        return player_df
    
# using multiprocess to speed up scraping and data transformation -- 9m50s

links = get_links()
with Pool(10) as p:
    records = p.map(scrape_player, links)
    
all_player_stats = pandas.DataFrame()
for record in records:
    all_player_stats = pandas.concat([all_player_stats,record], axis=0, ignore_index=True)
    
all_player_stats.sample(100)
# all_player_stats.to_csv("dirty_aba_nba_players.csv")

,Player Name,Season,Age,Tm,Lg,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Trp Dbl,Unnamed: 18,Unnamed: 22,Unnamed: 23,Unnamed: 28,Unnamed: 29
30790,Tayshaun Prince,2004-05,24.0,DET,NBA,SF,82.0,82.0,3039.0,469.0,963.0,0.487,47.0,138.0,0.341,422.0,825.0,0.512,0.511,221.0,274.0,0.807,133.0,302.0,435.0,247.0,56.0,71.0,135.0,161.0,1206.0,NaN,NaN,NaN,NaN,NaN,NaN
35346,Eric Snow,2000-01,27.0,PHI,NBA,PG,50.0,50.0,1740.0,182.0,435.0,0.418,5.0,19.0,0.263,177.0,416.0,0.425,0.424,122.0,154.0,0.792,27.0,139.0,166.0,369.0,77.0,7.0,124.0,123.0,491.0,0.0,NaN,NaN,NaN,NaN,NaN
15677,Udonis Haslem,2008-09,28.0,MIA,NBA,PF,75.0,75.0,2560.0,333.0,643.0,0.518,0.0,0.0,NaN,333.0,643.0,0.518,0.518,128.0,170.0,0.753,169.0,449.0,618.0,85.0,43.0,25.0,82.0,204.0,794.0,NaN,NaN,NaN,NaN,NaN,NaN
24016,Don Martin,1946-47,26.0,STB,BAA,NaN,54.0,NaN,NaN,89.0,304.0,0.293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.0,31.0,0.419,NaN,NaN,NaN,9.0,NaN,NaN,NaN,75.0,191.0,NaN,NaN,NaN,NaN,NaN,NaN
35559,Elmore Spencer,1995-96,26.0,POR,NBA,C,1.0,0.0,1.0,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,NaN,0.0,2.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
21196,Travis Knight,2002-03,28.0,NYK,NBA,C,32.0,0.0,287.0,25.0,65.0,0.385,0.0,1.0,0.000,25.0,64.0,0.391,0.385,10.0,13.0,0.769,18.0,44.0,62.0,14.0,8.0,9.0,10.0,44.0,60.0,NaN,NaN,NaN,NaN,NaN,NaN
2653,Byron Beck,1970-71,26.0,DNR,ABA,PF,84.0,NaN,2849.0,490.0,1033.0,0.474,4.0,14.0,0.286,486.0,1019.0,0.477,0.476,158.0,182.0,0.868,280.0,604.0,884.0,177.0,NaN,NaN,160.0,273.0,1142.0,NaN,NaN,NaN,NaN,NaN,NaN
14458,Kevin Grevey,1975-76,22.0,WSB,NBA,SF,56.0,NaN,504.0,79.0,213.0,0.371,NaN,NaN,NaN,79.0,213.0,0.371,0.371,52.0,58.0,0.897,24.0,36.0,60.0,27.0,13.0,3.0,NaN,65.0,210.0,NaN,NaN,NaN,NaN,NaN,NaN
15888,Tom Hawkins,1967-68,31.0,LAL,NBA,PF,78.0,NaN,2463.0,389.0,779.0,0.499,NaN,NaN,NaN,NaN,NaN,NaN,NaN,125.0,229.0,0.546,NaN,NaN,458.0,117.0,NaN,NaN,NaN,289.0,903.0,NaN,NaN,NaN,NaN,NaN,NaN
40707,John Williams,1989-90,23.0,WSB,NBA,SF,18.0,18.0,632.0,130.0,274.0,0.474,2.0,18.0,0.111,128.0,256.0,0.500,0.478,65.0,84.0,0.774,27.0,109.0,136.0,84.0,21.0,9.0,43.0,33.0,327.0,0.0,NaN,NaN,NaN,NaN,NaN
